In [21]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler

# Load your data
df = pd.read_csv("/Users/kimballweeks/Downloads/cleaned_data.csv")

# Rename for consistency
df = df.rename(columns={
    "pct_highschool_or_more (1990)": "pct_hs_1990",
    "pct_highschool_or_more_(2023)": "education_2022",
    "income_1989_real_2023": "income_1989_real",
    "Pop 2023": "pop_2023"
})

# Clean up whitespace in State names
df['State'] = df['State'].astype(str).str.strip()

# Convert relevant columns to numeric (safely)
cols_to_numeric = [
    'church_persistence_index',
    'income_1989_real',
    'pct_hs_1990',
    'pop_2023',
    'education_2022'
]
df[cols_to_numeric] = df[cols_to_numeric].apply(pd.to_numeric, errors='coerce')

# Drop rows with missing values in any key columns
df = df.dropna(subset=cols_to_numeric + ['State'])

# Log-transform population
df['log_pop_2023'] = np.log(df['pop_2023'])

# Standardize controls
scaler = StandardScaler()
df[['income_1989_scaled', 'pct_hs_1990_scaled']] = scaler.fit_transform(
    df[['income_1989_real', 'pct_hs_1990']]
)

# Run the regression
model = smf.ols(
    formula='education_2022 ~ church_persistence_index + income_1989_scaled + pct_hs_1990_scaled + log_pop_2023 + C(State)',
    data=df
).fit(cov_type='HC3')

# Output summary
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:         education_2022   R-squared:                       0.653
Model:                            OLS   Adj. R-squared:                  0.647
Method:                 Least Squares   F-statistic:                     113.5
Date:                Thu, 09 Oct 2025   Prob (F-statistic):               0.00
Time:                        11:47:51   Log-Likelihood:                -7895.2
No. Observations:                3055   AIC:                         1.589e+04
Df Residuals:                    3003   BIC:                         1.621e+04
Df Model:                          51                                         
Covariance Type:                  HC3                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               